                               Restaurant Rating Prediction and Classification using Machine Learning

Problem Statement:

The restaurant industry, particularly in metropolitan cities like Bengaluru, has witnessed exponential growth, with thousands of restaurants competing for customer attention. Zomato, being one of the largest restaurant aggregator and food delivery platforms, hosts extensive data on restaurants including their location, cuisine type, cost, customer votes, and ratings.

For new and existing restaurant owners, understanding what factors influence customer ratings is critical for making informed business decisions — such as choosing a location, cuisine type, pricing strategy, or service features (like online ordering or table booking). Manually analyzing thousands of records to identify these patterns is impractical.

This project aims to solve this problem by applying machine learning techniques to the Zomato Bangalore Restaurants dataset to predict restaurant ratings (regression) and classify restaurants into rating categories (classification), thereby providing actionable insights for stakeholders in the food service industry.

Objectives

1)To perform data cleaning and preprocessing on the raw Zomato dataset to handle missing values, inconsistent formatting, and irrelevant features.
2)perform Exploratory Data Analysis (EDA) to understand relationships between features such as location, cuisine, cost, and rating.
3)apply feature scaling (StandardScaler) to normalize numerical features for model compatibility.
4) build a regression model to predict the exact restaurant rating (continuous value).
5)build a classification model to categorize restaurants into rating classes (e.g., Excellent, Good, Average, Poor).
6)implement and compare multiple machine learning algorithms, including Decision Tree, Naive Bayes, Linear Regression, and Logistic Regression.
7)perform hyperparameter tuning using GridSearchCV/RandomizedSearchCV to optimize model performance.
To evaluate and compare model performance using appropriate metrics (RMSE, R², Accuracy, Precision, Recall, F1-score).
To draw meaningful business insights that can help restaurant owners and stakeholders make data-driven decisions.

Scope of the Project

The project is limited to restaurants listed in Bengaluru as per the dataset used.

The analysis focuses on structured/tabular data (location, cost, cuisine, votes, rating, etc.); text-based features like customer reviews are excluded from the core ML pipeline (optional extension for future work).

The project covers supervised learning techniques only regression and classification are used.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR


In [2]:
df = pd.read_csv("zomato.csv")  

df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [3]:
df.isnull().sum()

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            7775
votes                              0
phone                           1208
location                          21
rest_type                        227
dish_liked                     28078
cuisines                          45
approx_cost(for two people)      346
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          51717 non-null  object
 1   address                      51717 non-null  object
 2   name                         51717 non-null  object
 3   online_order                 51717 non-null  object
 4   book_table                   51717 non-null  object
 5   rate                         43942 non-null  object
 6   votes                        51717 non-null  int64 
 7   phone                        50509 non-null  object
 8   location                     51696 non-null  object
 9   rest_type                    51490 non-null  object
 10  dish_liked                   23639 non-null  object
 11  cuisines                     51672 non-null  object
 12  approx_cost(for two people)  51371 non-null  object
 13  reviews_list                 51

In [5]:
df = df.drop(['url', 'dish_liked', 'phone'], axis=1)
df.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'location', 'rest_type', 'cuisines', 'approx_cost(for two people)',
       'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)'],
      dtype='object')

In [6]:
df.rename({
    'approx_cost(for two people)': 'cost_for_2',
    'listed_in(type)': 'listed_in_type',
    'listed_in(city)': 'listed_in_city'
}, axis=1, inplace=True)
df.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'location', 'rest_type', 'cuisines', 'cost_for_2', 'reviews_list',
       'menu_item', 'listed_in_type', 'listed_in_city'],
      dtype='object')

In [7]:
df.isnull().sum()


address              0
name                 0
online_order         0
book_table           0
rate              7775
votes                0
location            21
rest_type          227
cuisines            45
cost_for_2         346
reviews_list         0
menu_item            0
listed_in_type       0
listed_in_city       0
dtype: int64

In [8]:
remove_comma = lambda x: int(x.replace(',', '')) if isinstance(x, str) and x != np.nan else x

In [9]:
df.votes =df.votes.astype('int')
df['approx_cost_for_2_people'] = df['approx_cost_for_2_people'].apply(remove_comma)

df.info()                   #checking if anu null values is there or not

KeyError: 'approx_cost_for_2_people'

we can see by above operations that the data is clean and ready to use removing null and duplicated values in the dataset taken fro zomato  

In [ ]:
df['rate'].unique()


In [ ]:
df=df.loc[df.rate!='NEW']           # removing junk values like new and- in the data
df=df.loc[df.rate!='-'].reset_index(drop=True)#removing messy data by deleting ows =NEW

In [ ]:
remove_slash=lambda x:x.replace('/5','')if isinstance(x,str)else x  #converting text to real nos
df.rate=df.rate.apply(remove_slash).str.strip().astype('float')

In [ ]:
df['rate'].head()

In [ ]:
df.info()

In [ ]:
#label encoding
def Encode(df):  #turns text columns into numeric codes using factorize
    for column in df.columns[~df.columns.isin(['rate', 'approx_cost_for_2_people', 'votes'])]:
        df[column] = df[column].factorize()[0]
    return df

df_en = Encode(df.copy())  #encodes a copy of old redable data

In [ ]:
df_en['rate'] = df_en['rate'].fillna(df_en['rate'].mean()) #fill missing values by taking mean
df_en['approx_cost_for_2_people'] = df_en['approx_cost_for_2_people'].fillna(df_en['approx_cost_for_2_people'].mean())


In [ ]:
df_en.isna().sum()

In [ ]:
corr=df_en.corr(method='kendall')
plt.figure(figsize=(50,40))
sns.heatmap(corr,annot=True)
plt.title('kendall corelation heatmap')
plt.show()

In [ ]:
x=df_en.iloc[:,[2,3,4,5,6,7,8,9,11]]
y=df_en['rate']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=353)


In [ ]:
x_train.head()
y_train.head()